# Dive.ai 서비스 파이프라인 초안(변경 가능)

사용자가 세계관·소재를 입력하면 AI가 시나리오를 생성하고, 분기형 인터랙티브 서사로 변환하여 엔딩까지 도달하는 전체 흐름을 정리합니다.

## 전체 구조 개요

3개의 핵심 기능을 중심으로 구성됩니다.

```
[시나리오 빌더] → [시나리오 트랜스포머] → [인터랙티브 챗 & 플레이] → [엔딩]
                          ↕
              로어북 (전 단계에 걸쳐 작동하는 서포트 레이어)
```

### 기획 배경

- **AI 캐릭터 채팅 시장의 급성장**: 전 세계적으로 AI 캐릭터 대화 서비스가 폭발적으로 성장하고 있으나, 대다수 서비스가 '단순 일상 대화'에 그쳐 콘텐츠의 휘발성 문제와 사용자가 직접 대화를 이끌어 나가야 한다는 부담감이 있음.
- **타인 의존적 콘텐츠 소비 구조**: 현재 서비스는 타인이 만들어 놓은 페르소나·세계관에 유저가 '손님'으로 참여하는 형태에 치중. 사용자가 직접 시나리오와 캐릭터를 만들고, 본인이 만든 시나리오 속에서 캐릭터와 소통하는 '서사의 주인공이자 창작자' 경험을 목표로 함.

### 문제 해결 방향

| 방향 | 내용 |
|------|------|
| 대화에서 '플레이'로 | 시나리오 빌더를 통해 '엔딩'이라는 명확한 목표를 가진 인터랙티브 서사 구조 구축 |
| 선형 서사의 입체화 | 사용자의 소재를 LLM이 기승전결로 확장하고, 다시 분기형 인터랙티브 구조로 변환 |
| 상황 인식형 멀티모달 | 분기점 선택 시 대화 맥락·캐릭터 감정 상태를 분석하여 실시간 AI 이미지 생성 |

## 0단계 — 세계관 & 소재 입력

두 가지 진입 경로를 제공합니다.

```
두 가지 진입 경로
    │
    ├─► 경로 A: 기존 세계관 선택
    │     콘텐츠 유형 선택
    │     만화 / 시리즈 / 영화 / 소설 / 고전
    │         │
    │         ├─► 만화 / 시리즈 / 영화 / 소설
    │         │     장르 선택
    │         │     드라마 / 멜로·로맨스 / 스릴러 / 판타지
    │         │     액션 / 미스터리 / 코미디 / SF / 전쟁
    │         │     공포(호러) ← 소설 선택 시 비표시 ⚠️
    │         │         └─► 작품명 검색 또는 직접 입력
    │         │
    │         └─► 고전
    │               ① 국가 선택
    │               한국 / 중국 / 일본
    │                   │
    │                   ② 선택한 국가의 장르 목록 표시
    │                   │
    │                   ├─► 한국 선택 시
    │                   │     가문소설 / 판타지 / 로맨스
    │                   │     영웅소설 / 미스터리 / 호러
    │                   │
    │                   ├─► 중국 선택 시
    │                   │     판타지 / 로맨스 / 무협
    │                   │     호러 / 미스터리
    │                   │
    │                   └─► 일본 선택 시
    │                         설화 / 미스터리 / 호러
    │                         편지소설 / 영험담
    │                   │
    │                   ③ 장르 선택 (미선택 시 해당 국가 전체 검색)
    │                       └─► 작품명 검색 또는 직접 입력
    │
    └─► 경로 B: 직접 소재 입력
          유저가 키워드 / 아이디어 / 소재 직접 입력
          장르 선택
          → AI가 세계관 자동 확장
```

### 유형별 선택 가능 장르 (`CATEGORY_TO_GENRES`)

소설 카테고리에 `공포(호러)` 데이터가 없으므로, 소설 선택 시 해당 장르를 프론트엔드에서 노출하지 않습니다.

| 유형 | 장르 선택지 |
|------|------------|
| 영화 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |
| 시리즈 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |
| 소설 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 전쟁 **(공포(호러) 제외)** |
| 만화 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |

### 국가별 선택 가능 장르 (`COUNTRY_TO_GENRES`)

| 국가 | 장르 선택지 |
|------|------------|
| 한국 | 가문소설 · 판타지 · 로맨스 · 영웅소설 · 미스터리 · 호러 |
| 중국 | 판타지 · 로맨스 · 무협 · 호러 · 미스터리 |
| 일본 | 설화 · 미스터리 · 호러 · 편지소설 · 영험담 |

## 핵심 기능 1 — AI 시나리오 & 캐릭터 빌더

사용자가 입력한 소재·캐릭터 정보를 바탕으로 기승전결 시나리오와 AI 캐릭터 초기 이미지를 생성합니다.

- **데이터**: 문화콘텐츠 스토리 데이터, 동아시아 고전 스토리 데이터, 직접 수집한 웹소설 데이터
- **기술**: LLM + RAG. 장르별 흥행 서사 패턴을 반영하여 복선과 갈등 구조가 설계된 트리트먼트 작성

```
유저 입력 (장르·테마·소재와 함께 동시 입력)
    │
    ├─► AI 캐릭터 정보 (유저와 대화할 상대방 캐릭터)
    │     이름 / 성격 / 외형 / 배경
    │
    └─► 유저 페르소나 정보 (유저를 대신할 캐릭터)
          이름 / 성격 / 배경
    │
    ▼
RAG 검색
    ├─► scenes 컬렉션      (문화콘텐츠 스토리 데이터, 89,851 docs)
    │     필터: category, genre, arc_stage
    ├─► classics 컬렉션    (동아시아 고전 데이터, 12,717 docs)
    │     필터: country, classic_genre
    └─► 웹소설 데이터       (직접 수집)
          장르별 흥행 서사 패턴 / 복선 구조 / 갈등 패턴 참조
    │
    ▼
시나리오 생성 (LLM + RAG)
    두 캐릭터를 중심으로 기승전결 트리트먼트 작성
    ├─► 복선 설계 (기 단계에 심고 결 단계에서 유저 선택으로 회수)
    ├─► 갈등 구조 설계
    ├─► 주요 사건 목록
    ├─► [서사의 씨앗(복선)] 출력
    └─► [예상되는 멀티 엔딩 조건] 출력
    ※ 시나리오는 유저에게 노출하지 않고 백엔드 진행 가이드로만 사용
       (개발 단계에서는 출력하여 품질 확인 및 프롬프트 보완)
    │
    ▼
AI 캐릭터 초기 이미지 생성 (기준 이미지)
    AI 캐릭터 외형 설명 → 이미지 생성 (이미지 API 미정)
    → 기준 이미지 저장
      이후 승·전·결 단계별 이미지 생성 시 얼굴 레퍼런스로 고정
    │
    ▼
로어북 자동 초기화
    시나리오에서 핵심 항목 자동 추출 → Vector DB 색인화
    ┌──────────────────────────────────────────┐
    │ 인물 항목   이름, 특징, 관계, trigger      │
    │ 장소 항목   지명, 설명, trigger            │
    │ 사건 항목   사건명, 경과, 결과, trigger     │
    │ 설정 항목   고유 명사, 역사적 사건, trigger  │
    └──────────────────────────────────────────┘
    (이벤트 스트리밍 & 자동 인덱싱)
```

## 핵심 기능 2 — AI 시나리오 트랜스포머

선형 시나리오를 인터랙티브 분기점(Choice Points)으로 변환합니다.

- **기술**: LLM을 활용한 Narrative Branching — 흥행 콘텐츠의 갈등 구조를 학습하여 몰입감 있는 선택지 생성

```
확정된 선형 시나리오
    │
    ▼
분기형 인터랙티브 구조로 변환 (LLM)
    흥행 콘텐츠의 갈등 구조 패턴 학습 기반 (Narrative Branching)
    │
    ▼
분기 트리 구조 설계
    │
    ├─► 분기점(Choice Points) 생성
    │     갈등 절정부에 선택지 배치
    │     선택지 2~3개 (서사적으로 의미 있는 차이)
    │
    ├─► 분기별 전개 방향 설계
    │     선택 A → 전개 방향 A (캐릭터 관계 변화, 사건 흐름 변화)
    │     선택 B → 전개 방향 B
    │     분기간 교차 경로 (특정 조건 시 다른 루트로 합류)
    │
    └─► 엔딩 조건 설계
          ┌──────────────────────────────────────┐
          │ 해피 엔딩  호감도 + 선택 플래그 조합    │
          │ 배드 엔딩  특정 플래그 미달성           │
          │ 트루 엔딩  전체 복선 해소 + 조건 충족   │
          │ 루트 엔딩  특정 분기 경로 완주          │
          └──────────────────────────────────────┘
```

## 핵심 기능 3 — 인터랙티브 챗 & 플레이

### 플레이 시작 전 설정

| 유저 페르소나 | 유저 노트 | 세션 옵션 |
|-------------|---------|----------|
| 이름/성격/배경 | AI가 항상 기억할 사항을 유저가 직접 작성 | 출력 모델 (Gemini Flash/Pro) |
| 직업/외모 | (출력 빈도 최소화) | 추론 양 (저/중/고) |
| 적용/수정/삭제 | | 감성 (긍정/중립/부정) |
| | | AI 주도 사건 (on/off) |
| | | 사칭 설정 (on/off) |
| | | 시작 설정 (커스텀/AI추천/랜덤) |

### 대화 턴 (분기점 전: 자유 대화)

```
유저 입력
    ├─► "" 형식     → 캐릭터 대화로 인식
    ├─► 서술 형식   → 유저 행동/상황으로 인식
    └─► ! 명령어
          !요약  → 즉시 요약 후 저장
          !설정  → 추가 세계관 설정
    │
    ▼
시스템 프롬프트 구성
    ├─► 고정 레이어 (항상 포함)
    │     유저 페르소나 + 유저 노트 + 감성 설정
    │
    ├─► 동적 레이어 A — 로어북
    │     현재 대화 마지막 3~5턴 → Semantic Search
    │     → 문맥적 의미 유사 항목 선별 (단순 키워드 일치 X)
    │     → 시스템 프롬프트에 실시간 주입 (유저 비노출)
    │     → 과거 사건 ↔ 현재 대화 충돌 감지 (Narrative Consistency Engine)
    │
    ├─► 동적 레이어 B — RAG 씬 컨텍스트
    │     현재 기승전결 단계 기반 scenes/classics 검색
    │     causality 있을 때만 씬 컨텍스트에 포함 (씬 간 흐름 힌트)
    │
    └─► 동적 레이어 C — 요약 기억
          10~15턴 초과 or !요약 시 생성된 요약본 주입
    │
    ▼
AI 생성 (선택된 Gemini 모델)
    ├─► 캐릭터 대화 + 행동 묘사
    ├─► 상태창 업데이트 (호감도 / 친밀도 / 캐릭터 내면 생각)
    └─► 대화 추천 선택지 3개
```

### 분기점 도달

```
AI가 분기점 조건 충족 감지
    │
    ▼
선택지 제시 (2~3개)
    각 선택지는 서사적으로 다른 방향을 만드는 의미 있는 차이
    │
    ▼
유저 선택
    │
    ▼
분기 상태 저장 (선택 플래그 기록)
    │
    ├─► AI 캐릭터 단계별 이미지 생성 (이미지 API 미정)
    │     기준 이미지 (초기 생성본) → 얼굴 레퍼런스로 고정
    │     + 현재 분기 상황 묘사 (배경 / 의상 / 표정 / 상황)
    │     → 얼굴 일관성 유지, 주변 환경·상황만 변화
    │     → 이미지 출력 (채팅창 인라인)
    │
    └─► 해당 분기 시나리오로 대화 계속
```

### 이미지 생성 흐름 (AI 캐릭터)

```
[최초 생성 — 기 단계]
AI 캐릭터 외형 설명 → 기준 이미지 생성 → 저장

[승 단계 진입]
기준 이미지 (얼굴 고정) + 승 단계 상황 묘사
→ 얼굴 유지 / 배경·의상·표정 변화

[전 단계 진입]
기준 이미지 (얼굴 고정) + 전 단계 상황 묘사
→ 얼굴 유지 / 배경·의상·표정 변화

[결·엔딩]
기준 이미지 (얼굴 고정) + 엔딩 상황 묘사
→ 얼굴 유지 / 배경·의상·표정 변화

※ 이미지 생성 API 및 얼굴 일관성 유지 방식 미정
   (Leonardo.ai Character Reference / Civitai IP-Adapter 등 검토 예정)
```

### 분기 내 진행

```
선택한 분기의 흐름으로 대화 계속
    │
    ├─► 로어북 실시간 업데이트
    │     분기별 새 사건/설정 자동 추출 → 색인 추가 (on/off)
    │
    ├─► 캐릭터 상태 분기별 독립 저장
    │     퇴장한 캐릭터 상태 보존 → 재등장 시 불러오기
    │
    └─► 다음 분기점 or 엔딩 조건 감지 대기
```

## 엔딩 도달

```
엔딩 조건 판단
    ├─► 호감도/친밀도 누적값
    ├─► 선택 이력 플래그 조합
    └─► 특정 사건 달성 여부
    │
    ▼
엔딩 타입 결정
    해피 엔딩 / 배드 엔딩 / 트루 엔딩 / 루트별 엔딩
    │
    ▼
엔딩 씬 생성 (LLM)
    │
    ▼
엔딩 이미지 생성 (이미지 API 미정)
    기준 이미지 (얼굴 고정) + 엔딩 상황 묘사
    → 얼굴 일관성 유지, 엔딩 분위기에 맞는 환경·표정 반영
    │
    ▼
엔딩 결과 화면
    ├─► 엔딩 명칭 + 엔딩 씬 텍스트 + 이미지
    ├─► 달성한 분기 경로 표시
    ├─► 다른 엔딩 힌트 (선택적)
    └─► 공유 / 저장 / 재도전 버튼
```

## 서포트 레이어 — 로어북

사용자가 세계관의 핵심 설정(지명, 인물 관계, 고유 명사, 역사적 사건 등)을 등록·관리하며, AI가 대화 중 이를 실시간으로 참조하여 서사의 일관성을 유지합니다.

```
┌──────────────────────────────────────────────────────────┐
│                       로어북                              │
│                                                          │
│  [시나리오 빌더 단계]                                      │
│    시나리오 생성 완료 → 주요 항목 자동 추출 → Vector DB 색인  │
│    이벤트 스트리밍 & 자동 인덱싱                             │
│                                                          │
│  [챗 단계 - 매 턴]                                        │
│    현재 대화 문맥 → Semantic Search                        │
│    → 문맥적 의미 유사 항목 선별 (단순 키워드 일치 X)           │
│    → 시스템 프롬프트에 동적 결합                             │
│    Real-time Context Injection                           │
│                                                          │
│  [서사 일관성 유지]                                        │
│    과거 사건 ↔ 현재 대화 충돌 감지                           │
│    AI 기억 우선순위 관리                                   │
│    Narrative Consistency Engine                          │
│                                                          │
│  [유저 수동 관리]                                         │
│    지명 / 인물 관계 / 고유 명사 / 역사적 사건                 │
│    항목 추가 / 수정 / 삭제                                  │
└──────────────────────────────────────────────────────────┘
```

### 유저노트&로어북

| 항목 | 유저노트 | 로어북 |
|------|---------|--------|
| **작성 주체** | 유저가 직접 수동 작성 | 자동 추출 + 유저 수동 등록 |
| **컨텍스트 포함 방식** | 항상 전체 내용이 시스템 프롬프트에 포함 | 대화 문장과 의미적으로 유사한 항목만 선별 주입 |
| **구조** | 자유 텍스트 (메모장) | 항목별 구조화 (키워드 + 설명) |
| **토큰 소모** | 매 턴마다 전체 소모 | 필요한 항목만 소모 (효율적) |
| **업데이트** | 유저가 직접 수정 | 시나리오 확정 시 자동 추출 + 채팅 중 실시간 추가 |
| **주요 용도** | "AI야, 이건 항상 기억해" (유저 주도 고정 지시) | 세계관·사건·인물 설정 일관성 유지 (서사 주도) |

## 세션 & 계정 관리

```
구글 로그인 → PC / 모바일 연동
    │
    ├─► 다중 세션 (+ 버튼 → 새 대화창, 이름 수정/삭제)
    │
    ├─► 대화 수정 / 공유 / 삭제
    │     수정: 해당 턴 이후 삭제 → 해당 시점부터 재진행
    │     공유: 해당 시점부터 독립 세션 분기
    │
    └─► 엔딩 아카이브 (달성한 엔딩 목록 저장)
```

## 전체 데이터 흐름 요약

```
[세계관 선택 or 소재 직접 입력]
    ↓
[시나리오 빌더: RAG + LLM → 기승전결 + 캐릭터 생성]
    ↓
[로어북 자동 초기화: 핵심 항목 추출 → Vector DB 색인]
    ↓
[시나리오 트랜스포머: 선형 → 분기 구조 + 엔딩 조건 설계]
    ↓
[플레이 시작: 유저 페르소나 + 세션 설정]
    ↓
[인터랙티브 챗: 로어북 + RAG + 요약기억 → 매 턴 대화]
    ↓
[분기점 도달 → 선택 → 이미지 생성 → 분기 진입]
    ↓
[엔딩 조건 달성 → 엔딩 씬 + 이미지 → 결과 화면]
```

## 개발 로드맵

### 현재 완료된 것

| 항목 | 파일 | 비고 |
|------|------|------|
| ChromaDB 벡터 DB 구축 | 벡터DB.ipynb §1~4 | 문화콘텐츠 + 동아시아 고전 씬 임베딩 |
| 기승전결 스테이지 매핑 | 벡터DB.ipynb §6 | 스토리헬퍼 16단계 → 기/승/전/결 매핑 |
| RAG 검색 + 프롬프트 빌더 | 벡터DB.ipynb §7~8 | 장르·카테고리·story_arc 기반 검색 |
| 시나리오 생성 함수 v1 | 벡터DB.ipynb §10 Cell 28 | AI/유저 캐릭터, 복선 설계, 멀티엔딩 조건 포함 |

---

### 남은 작업 순서

#### 1단계 — 시나리오 빌더 마무리 (핵심 기능 1)
- [ ] `generate_scenario_with_rag` 프롬프트 최종 정리
  - 소재 스케일 유지 원칙 반영
  - 복선을 기승전결 본문에 직접 삽입 강제
- [ ] 로어북 자동 초기화: 시나리오 생성 후 핵심 항목(지명·인물·사건) 자동 추출 → ChromaDB 색인

#### 2단계 — 시나리오 트랜스포머 (핵심 기능 2)
- [ ] 선형 기승전결 → 분기 구조 변환 로직 설계
- [ ] Choice Points + 분기 트리 자료구조 정의
- [ ] 엔딩 조건 설계 (호감도 누적값, 선택 이력 플래그 조합)

#### 3단계 — 인터랙티브 챗 & 플레이 (핵심 기능 3)
- [ ] 유저 페르소나 + 세션 옵션 설정 시스템
- [ ] 대화 턴 시스템
  - 고정 레이어 (유저 페르소나 + 유저 노트 + 감성 설정)
  - 동적 레이어 A (로어북 실시간 참조)
  - 동적 레이어 B (요약 기억: 대화 압축 + 분기 플래그)
- [ ] 분기점 감지 + 선택지 제시 + 선택 플래그 기록

#### 4단계 — 엔딩
- [ ] 엔딩 조건 판단 로직 (호감도·플래그·사건 달성 여부 조합)
- [ ] 엔딩 씬 생성 (LLM)
- [ ] 엔딩 이미지 생성 (이미지 API 미정)

#### 5단계 — 로어북 자동화
- [ ] 대화 중 로어북 키워드 감지 + 실시간 컨텍스트 삽입
- [ ] 유저 직접 등록·수정·삭제 인터페이스

#### 6단계 — 세션 & 계정 관리
- [ ] 구글 로그인 + PC/모바일 연동
- [ ] 다중 세션, 대화 수정/공유/삭제
- [ ] 엔딩 아카이브

---

### 오늘 할 작업 (2026-04-24)

1. **`generate_scenario_with_rag` 프롬프트 최종 정리** (벡터DB.ipynb Cell 28)
   - 현재 system_prompt 중복 구조 정리
   - 소재 스케일 유지 원칙 + 복선 본문 삽입 강제 규칙 확정 반영
2. **시나리오 트랜스포머 구조 설계 시작** (파이프라인.ipynb 또는 신규 노트북)
   - 선형 기승전결을 받아 분기 트리로 변환하는 자료구조 정의
   - Choice Point 조건 추출 방식 설계
